# Notebook 1: Your first tool-use loop

A model is good with words but cannot do exact arithmetic on its own. So we give
it a **calculator tool**. You will watch the model **ask** to use the calculator,
your code **run** it, you **return** the answer, and the model **continue**.

That four step rhythm (ask, run, return, continue) is the heart of tool use.

## Setup: run these first

Every notebook starts the same way: install the library, load your key, and
pick a model. Run the next three cells in order.

**Save yourself time:** store your API key once in Colab Secrets (the key icon on
the left sidebar). Add a secret named `ANTHROPIC_API_KEY`, paste your key as the
value, and turn on "Notebook access". After that, every notebook picks it up
automatically and you never paste it again.

In [ ]:
# 1) Install the Anthropic library (Colab forgets it between sessions, so we
#    install each time). The -q just keeps the output quiet.
!pip install -q anthropic

In [ ]:
# 2) Load your Anthropic API key.
#    Easiest: store it once in Colab Secrets (key icon on the left), named
#    ANTHROPIC_API_KEY, with "Notebook access" on. Then this reads it for you.
import anthropic

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = None  # not on Colab, or no secret set yet

if not api_key:
    # Fallback: ask once if no secret is found. The input is hidden and not saved.
    from getpass import getpass
    api_key = getpass("Paste your Anthropic API key (input is hidden): ")

client = anthropic.Anthropic(api_key=api_key)
print("Your client is ready.")

In [ ]:
# 3) Choose the model. claude-opus-4-8 is the most capable. For faster, cheaper
#    runs while practising you can switch this to "claude-haiku-4-5".
MODEL = "claude-opus-4-8"
print("Using model:", MODEL)

## Step 1: define the tool

A tool has a name, a plain description, and the inputs it needs. The model reads
the description to decide when to use it. We also write the actual Python that
does the work.

In [ ]:
import ast
import operator

# A tool is described by a name, a plain description, and the inputs it needs.
# The model reads the description to decide when to use the tool.
calculator_tool = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression and return the result. Use this whenever the user asks for a calculation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "An arithmetic expression, for example '12 * 37'.",
            }
        },
        "required": ["expression"],
    },
}

# The safe operations our calculator allows: plus, minus, times, divide, and
# negative numbers. Nothing else is permitted.
_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    # Walk the parsed expression and only allow numbers and the safe operations
    # above. Anything else (names, function calls) raises an error, so no
    # arbitrary code can run. This is the safe alternative to eval().
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")


def run_calculator(expression):
    # Beginners can treat this as a black box: give it an expression, get the
    # result. It parses the text and evaluates only safe arithmetic, so it
    # cannot run arbitrary code.
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_eval(tree))
    except Exception:
        return "Error: I could only handle a simple arithmetic expression like '12 * 37'."

## Step 2: ask a question that needs the tool

We send the question along with the list of tools. If the model wants the
calculator, the reply will have `stop_reason` equal to `"tool_use"`.

In [ ]:
tools = [calculator_tool]

messages = [
    {"role": "user", "content": "What is 12 times 37? Use the calculator."}
]

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
)

print("Stop reason:", response.stop_reason)
for block in response.content:
    if block.type == "tool_use":
        print("The model wants to run:", block.name, "with", block.input)
    elif block.type == "text":
        print("The model said:", block.text)

## Step 3: run the tool and hand the result back

This is the important cell. We do four things:
1. Add the model's turn (which holds the tool request) to the conversation.
2. Run the requested tool in our own code.
3. Send the result back as a new turn.
4. Ask the model to continue, now that it has the answer.

In [ ]:
# YOUR TURN. Fill in the four steps. The instructor notebook has the full answer.

# 1) Add the model's turn (it holds the tool request) to the conversation.
messages.append({"role": "assistant", "content": response.content})

# 2) Find each tool request, run it, and collect the results.
tool_results = []
for block in response.content:
    if block.type == "tool_use":
        # TODO: call run_calculator with block.input["expression"] and save it as `result`.
        result = None  # replace this line
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": result,
        })

# 3) Hand the results back to the model as a new user turn.
messages.append({"role": "user", "content": tool_results})

# 4) Ask the model to continue now that it has the result.
followup = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
)
for block in followup.content:
    if block.type == "text":
        print("Model's final answer:", block.text)

## Recap

You just ran a full tool-use loop:

1. **Ask**: the model requested the calculator.
2. **Run**: your code did the math.
3. **Return**: you sent the result back.
4. **Continue**: the model finished the answer.

## Your turn

Change the question in Step 2 to a different calculation and run the cells again.
Then try a harder one that needs two calculations and see what happens.